In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from catboost import CatBoostClassifier


%matplotlib inline


# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
pathpath = os.path.join(path, 'Q3_data.csv')
df = pd.read_csv(pathpath)

In [ ]:
# Task 2: Write your code here:
display(df.head())

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

for col in df.columns:
    df[col] = df[col].fillna(df[col].mean())

check_missing_values(df)

In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

# No need to encode any categorical columns because there aren't any

In [ ]:
# Task 4: Write your code here:
ss = StandardScaler()
cols_to_scale = df.columns.drop("Target")

for column in cols_to_scale:
  df[column] = ss.fit_transform(df[[column]])


In [ ]:
# Task 5: Write your code here:
print(f"Default: {df['Target'].sum()}")
print(f"No Default: {(df['Target'] == 0).sum()}")

In [ ]:
# Task 1: Write your code here:
feature_cols = cols_to_scale = df.columns.drop("Target")
X = df[feature_cols]
y = df['Target']

In [ ]:
# Task 2,3,4,5: Write your code here:
n_splits = 5 # K

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
lr_accuracy = []
lr_f1 = []
model = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )

all_results = {}

all_results["Catboost"] = {'accuracy': [], 'f1': []}

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn model
  print(f"Training Catboost...")
  model.fit(X_train, y_train) # train
  y_pred = model.predict(X_test) # validate

  # 3. Save metrics for that model in this fold
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, zero_division=0)

  all_results["Catboost"]['accuracy'].append(accuracy)
  all_results["Catboost"]['f1'].append(f1)

print(f"  Accuracy:  {np.mean(all_results["Catboost"]['accuracy']):.4f}")
print(f"  F1-Score:  {np.mean(all_results["Catboost"]['f1']):.4f}")

In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

golden_feature = feature_importance.sort_values('importance', ascending=False).iloc[0, 0]

In [ ]:
# Task 2: Write your code here:
print(f"The golden feature is {golden_feature}!!!!")

In [ ]:
# Task Bonus: Write your code here: